# 02 — RAG simple avec citation des sources

Ce notebook construit et teste la première version de l'assistant :

1. **découpage** des articles en chunks, chacun portant le document, le numéro d'article, son titre et ses pages ;
2. **indexation** : chaque chunk est transformé en vecteur par le modèle d'embeddings `bge-m3` ;
3. **recherche** : pour une question, les k chunks les plus proches sont retrouvés ;
4. **génération** : le LLM `qwen3:4b-instruct` rédige une réponse à partir de ces seuls passages, en citant chaque affirmation par un numéro [n].

L'index doit avoir été construit au préalable avec `python scripts/build_index.py`.

In [1]:
import time

import pandas as pd
from llama_index.core.utils import get_tokenizer

from assistant_rag.chunking import build_chunks
from assistant_rag.config import DOCUMENTS, Settings
from assistant_rag.extraction import load_corpus, split_articles
from assistant_rag.rag import SimpleRAG

pd.set_option("display.max_colwidth", 90)
settings = Settings()
pd.Series(
    {
        "LLM": settings.llm_model,
        "Embeddings": settings.embed_model,
        "Taille des chunks (tokens)": settings.chunk_size,
        "Chevauchement (tokens)": settings.chunk_overlap,
        "Passages retrouvés (k)": settings.top_k,
    },
    name="valeur",
).to_frame()

,valeur
LLM,qwen3:4b-instruct
Embeddings,bge-m3
Taille des chunks (tokens),512
Chevauchement (tokens),64
Passages retrouvés (k),5


## 1. Découpage en chunks

Un article court forme un seul chunk. Un article plus long que la taille cible est redécoupé par phrases, avec un chevauchement entre deux morceaux consécutifs pour ne pas couper une idée en deux. Chaque chunk commence par le titre de son article, ce qui le situe pour la recherche comme pour le LLM.

In [2]:
articles = [article for pages in load_corpus().values() for article in split_articles(pages)]
nodes = build_chunks(articles, settings.chunk_size, settings.chunk_overlap)
tokenizer = get_tokenizer()

chunks = pd.DataFrame(
    [
        {
            "document": node.metadata["document_name"],
            "article": node.metadata["article"],
            "tokens": len(tokenizer(node.text)),
        }
        for node in nodes
    ]
)
chunks.groupby("document").agg(
    articles=("article", "nunique"),
    chunks=("article", "size"),
    tokens_médian=("tokens", "median"),
    tokens_max=("tokens", "max"),
)

,articles,chunks,tokens_médian,tokens_max
document,,,,
AI Act,113,244,454.0,547
RGPD,99,172,409.0,533


In [3]:
print(nodes[0].node_id)
print(nodes[0].metadata)
print()
print(nodes[0].text[:600])

rgpd-art001-00
{'document': 'rgpd', 'document_name': 'RGPD', 'reference': 'Règlement (UE) 2016/679', 'article': 1, 'title': 'Objet et objectifs', 'chapter': 'CHAPITRE I', 'start_page': 32, 'end_page': 32, 'chunk': 0, 'chunk_count': 1}

RGPD — Article 1 : Objet et objectifs
1. Le présent règlement établit des règles relatives à la protection des personnes physiques à l'égard du traitement des
données à caractère personnel et des règles relatives à la libre circulation de ces données.
2. Le présent règlement protège les libertés et droits fondamentaux des personnes physiques, et en particulier leur
droit à la protection des données à caractère personnel.
3. La libre circulation des données à caractère personnel au sein de l'Union n'est ni limitée ni interdite pour des
motifs liés à la protection des personnes physiques à l'éga


## 2. Chargement de l'index et du système

In [4]:
start = time.perf_counter()
rag = SimpleRAG.from_settings(settings)
print(f"Index chargé en {time.perf_counter() - start:.1f} s depuis {settings.index_dir.name}")

Index chargé en 12.5 s depuis bge-m3_c512_o64


## 3. Contrôle de la recherche

Avant de juger les réponses, on vérifie que la recherche retrouve le bon article pour quelques questions dont la réponse est connue. Le rang indique la position de l'article attendu parmi les passages retrouvés (1 = premier).

In [5]:
checks = [
    ("Quelles sont les conditions du droit à l'effacement des données ?", "rgpd", 17),
    ("Dans quels cas la désignation d'un délégué à la protection des données est-elle obligatoire ?", "rgpd", 37),
    ("Quel est le montant maximal des amendes administratives prévues par le RGPD ?", "rgpd", 83),
    ("Quelles pratiques en matière d'IA sont interdites par l'AI Act ?", "ai_act", 5),
    (
        "Quelles obligations de transparence s'appliquent aux systèmes d'IA qui interagissent avec des personnes ?",
        "ai_act",
        50,
    ),
    ("Quelles sanctions l'AI Act prévoit-il en cas de non-respect des pratiques interdites ?", "ai_act", 99),
]

rows = []
for question, document, article in checks:
    sources = rag.retrieve(question)
    ranks = [s.number for s in sources if s.document == document and s.article == article]
    rows.append(
        {
            "question": question,
            "attendu": f"{DOCUMENTS[document].name} art. {article}",
            "rang": ranks[0] if ranks else None,
            "articles retrouvés": ", ".join(f"{s.document_name} {s.article}" for s in sources),
        }
    )

retrieval = pd.DataFrame(rows)
print(f"Article attendu retrouvé : {retrieval['rang'].notna().sum()} / {len(retrieval)}")
retrieval

Article attendu retrouvé : 6 / 6


,question,attendu,rang,articles retrouvés
0,Quelles sont les conditions du droit à l'effacement des données ?,RGPD art. 17,1,"RGPD 17, RGPD 17, RGPD 7, RGPD 14, RGPD 6"
1,Dans quels cas la désignation d'un délégué à la protection des données est-elle obliga...,RGPD art. 37,1,"RGPD 37, RGPD 37, RGPD 39, RGPD 38, RGPD 27"
2,Quel est le montant maximal des amendes administratives prévues par le RGPD ?,RGPD art. 83,1,"RGPD 83, RGPD 83, RGPD 83, RGPD 83, AI Act 100"
3,Quelles pratiques en matière d'IA sont interdites par l'AI Act ?,AI Act art. 5,1,"AI Act 5, AI Act 5, AI Act 5, AI Act 5, AI Act 1"
4,Quelles obligations de transparence s'appliquent aux systèmes d'IA qui interagissent a...,AI Act art. 50,1,"AI Act 50, AI Act 50, AI Act 50, AI Act 13, AI Act 13"
5,Quelles sanctions l'AI Act prévoit-il en cas de non-respect des pratiques interdites ?,AI Act art. 99,1,"AI Act 99, AI Act 5, AI Act 100, AI Act 5, AI Act 5"


## 4. Génération des réponses

Pour chaque question : la réponse, les sources effectivement citées et le temps de réponse. La dernière question porte sur un sujet absent du corpus : le système doit refuser de répondre plutôt qu'inventer.

In [6]:
questions = [
    "Quelles sont les conditions du droit à l'effacement des données ?",
    "Quelles pratiques en matière d'IA sont interdites par l'AI Act ?",
    "Quel est le montant maximal des amendes administratives prévues par le RGPD ?",
    "Quelle est la durée légale du travail en France ?",
]

answers = []
for question in questions:
    answer = rag.answer(question)
    answers.append(answer)
    print(f"Question : {question}\n")
    print(answer.format())
    print(f"\nTemps de réponse : {answer.latency:.1f} s")
    print("-" * 100)

Question : Quelles sont les conditions du droit à l'effacement des données ?

Les conditions du droit à l'effacement des données sont les suivantes : les données doivent être effacées lorsque l'un des motifs suivants s'applique : elles ne sont plus nécessaires au regard des finalités pour lesquelles elles ont été collectées ou traitées ; la personne concernée retire son consentement et il n'existe pas d'autre fondement juridique au traitement ; la personne s'oppose au traitement et il n'existe pas de motif légitime impérieux ; les données ont fait l'objet d'un traitement illicite ; elles doivent être effacées pour respecter une obligation légale prévue par le droit de l'Union ou par le droit de l'État membre ; ou elles ont été collectées dans le cadre de l'offre de services de la société de l'information [1]. Ce droit ne s'applique pas lorsque le traitement est nécessaire à l'exercice du droit à la liberté d'expression, à respecter une obligation légale ou à exécuter une mission d'inté

In [7]:
pd.DataFrame(
    [
        {
            "question": a.question,
            "refus": a.refused,
            "sources citées": len(a.cited),
            "temps (s)": round(a.latency, 1),
        }
        for a in answers
    ]
)

,question,refus,sources citées,temps (s)
0,Quelles sont les conditions du droit à l'effacement des données ?,False,2,213.7
1,Quelles pratiques en matière d'IA sont interdites par l'AI Act ?,False,4,242.0
2,Quel est le montant maximal des amendes administratives prévues par le RGPD ?,False,1,184.1
3,Quelle est la durée légale du travail en France ?,True,0,91.1


## 5. Conclusions

**Recherche.** Pour les 6 questions de contrôle, l'article attendu est retrouvé en première position. Les articles longs, découpés en plusieurs chunks, occupent souvent plusieurs des 5 places retrouvées.

**Génération.** Les réponses reprennent fidèlement le texte des articles et chaque affirmation est citée. La question hors corpus est correctement refusée au lieu de recevoir une réponse inventée.

**Temps de réponse.** Sur processeur (8 Go de RAM, sans GPU), une réponse prend entre 1,5 et 4 minutes, l'essentiel du temps étant consacré à la lecture des passages par le LLM. Les évaluations complètes sont donc exécutées sur GPU.

**Limite.** Ces quelques exemples ne constituent pas une évaluation : la qualité est mesurée dans le notebook 03 sur 40 questions annotées.